# SNC - Thesis Results & Figures

Turns every evaluation metric collected across the model versions
(v1.0 -> v2.8) into publication-quality figures and a multi-sheet
Excel workbook, saved into a tidy folder tree on Drive for the thesis
and the jury presentation.

**Self-contained:** the metrics are embedded (sourced from
`docs/model_training_log.md` and the v2.8 run logs), so this notebook
needs no checkpoints, datasets, or GPU. It runs end-to-end in seconds.

## What it produces

`{DRIVE_ROOT}/thesis_results/`
- `figures/png/` - six 300-dpi PNGs (slides)
- `figures/pdf/` - the same six as vector PDFs (LaTeX/Word)
- `tables/snc_results.xlsx` - four sheets (version comparison,
  v2.8 detection, v2.8 SI-SDR, v2.8 training curve)

| Figure | Content |
|---|---|
| 1 | Detection F1 across versions (headline) |
| 2 | SI-SDRi across versions |
| 3 | v2.8 training curves (the head learns) |
| 4 | v2.8 detection F1 per class |
| 5 | v2.8 SI-SDRi per class |
| 6 | v2.8 precision vs recall scatter |

## 1. Setup - mount Drive, install libraries, output folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# DRIVE_ROOT is the one path to change if your Drive layout differs.
DRIVE_ROOT = '/content/drive/MyDrive/snc'

# matplotlib + pandas ship with Colab; openpyxl backs the .xlsx writer.
!pip install -q openpyxl

In [ ]:
# Imports, output-folder structure on Drive, and a save() helper.
import io
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# All artifacts land under a single tidy tree on Drive so the whole results
# set survives the Colab runtime and is easy to drop into the thesis:
#   {DRIVE_ROOT}/thesis_results/
#       figures/png/   300-dpi PNGs for slides
#       figures/pdf/   vector PDFs for the LaTeX/Word document
#       tables/        snc_results.xlsx
OUT = Path(DRIVE_ROOT) / "thesis_results"
FIG_PNG = OUT / "figures" / "png"
FIG_PDF = OUT / "figures" / "pdf"
TABLES = OUT / "tables"
for d in (FIG_PNG, FIG_PDF, TABLES):
    d.mkdir(parents=True, exist_ok=True)
print("Output tree:", OUT)

# A clean, publication-friendly style with a graceful fallback.
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 110

ACCENT, HILITE, MUTED = "#2563eb", "#dc2626", "#9ca3af"


def save(fig, name):
    """Write a figure as both 300-dpi PNG and vector PDF, then report."""
    fig.savefig(FIG_PNG / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_PDF / f"{name}.pdf", bbox_inches="tight")
    print(f"saved: {name}  (png + pdf)")

## 2. Curated evaluation metrics

The numbers below are transcribed from `docs/model_training_log.md`
(per-version summary) and the v2.8 Colab run logs (per-class detection,
per-class SI-SDR, and the 60-epoch training curve). Keeping them in one
place makes the figures fully reproducible and easy to audit.

In [ ]:
# Curated evaluation metrics.
# Provenance: docs/model_training_log.md (per-version summary) and the v2.8
# Colab run logs (per-class detection, per-class SI-SDR, training curve).
# This notebook is self-contained — it needs no checkpoints or datasets, so it
# reproduces every thesis figure in seconds.
import io
import numpy as np
import pandas as pd

# Version-level summary. NaN marks values never recorded (v1.0) or not
# meaningful (v2.0 broken, v2.5 invalid checkpoint, v2.7 detection collapse).
VERSION_ROWS = [
    ("v1.0", "ESC-50",     50, np.nan, np.nan, "baseline",  "FiLM U-Net established"),
    ("v2.0", "ESC+US",     56, np.nan, np.nan, "broken",    "Over-aggressive augmentation"),
    ("v2.1", "ESC+US",     56, -22.18, 0.21,   "ok",        "Fixed augmentation + inference norm"),
    ("v2.2", "ESC+US",     56, -22.79, 0.13,   "ok",        "All-encoder FiLM + multi-res L1"),
    ("v2.3", "ESC+US+FSD", 235, -21.49, 0.02,  "regressed", "FSD50K added (phantom FP)"),
    ("v2.4", "ESC+US+FSD", 200, -20.05, 0.09,  "ok",        "min_clips=40 floor + allow-list"),
    ("v2.6", "ESC+US+FSD", 227, -15.73, 0.17,  "ok",        "Detection head + hard negatives"),
    ("v2.7", "ESC+US+FSD", 227, np.nan, 0.00,  "collapse",  "Focal loss (gradient collapse)"),
    ("v2.8", "ESC+US",     56, -24.85, 0.32,   "best",      "BCE + clean 56-class vocab"),
]
summary_df = pd.DataFrame(VERSION_ROWS, columns=[
    "version", "dataset", "classes", "si_sdri_db", "detection_f1",
    "status", "key_change"])

# Per-class blocks: whitespace-delimited, no spaces in any value.
DETECTION_V28 = """
air_conditioner 8 0.00 0.00 0.00
airplane 9 0.16 0.33 0.21
breathing 6 0.25 0.50 0.33
brushing_teeth 8 0.50 0.75 0.60
can_opening 12 0.50 0.25 0.33
car_horn 10 0.30 0.60 0.40
cat 11 0.38 0.27 0.32
chainsaw 5 0.00 0.00 0.00
children_playing 4 0.11 0.25 0.15
chirping_birds 6 0.20 0.17 0.18
church_bells 5 0.18 0.40 0.25
clapping 9 0.80 0.44 0.57
clock_alarm 7 0.50 0.71 0.59
clock_tick 14 0.50 0.43 0.46
coughing 9 0.14 0.11 0.12
cow 14 0.57 0.29 0.38
crackling_fire 5 0.33 0.60 0.43
crickets 12 0.67 0.33 0.44
crow 7 0.33 0.57 0.42
crying_baby 9 0.60 0.67 0.63
dog 13 0.10 0.08 0.09
door_wood_creaks 12 0.00 0.00 0.00
door_wood_knock 3 0.00 0.00 0.00
drilling 11 0.50 0.18 0.27
drinking_sipping 9 0.20 0.11 0.14
engine 12 0.40 0.33 0.36
fireworks 12 0.21 0.50 0.30
footsteps 6 0.08 0.50 0.14
frog 13 1.00 0.31 0.47
glass_breaking 5 0.10 0.20 0.13
gun_shot 6 0.75 0.50 0.60
hand_saw 8 0.46 0.75 0.57
helicopter 11 0.80 0.36 0.50
hen 9 0.06 0.11 0.08
insects 7 0.40 0.29 0.33
jackhammer 7 0.29 0.57 0.38
keyboard_typing 7 0.22 0.57 0.32
laughing 7 0.00 0.00 0.00
mouse_click 10 0.25 0.20 0.22
pig 4 0.50 0.25 0.33
pouring_water 16 0.33 0.31 0.32
rain 12 0.55 0.50 0.52
rooster 3 0.14 0.33 0.20
sea_waves 10 0.60 0.30 0.40
sheep 7 0.33 0.86 0.48
siren 9 0.23 0.78 0.36
sneezing 9 0.14 0.11 0.12
snoring 9 0.12 0.22 0.16
street_music 12 0.35 0.58 0.44
thunderstorm 6 0.00 0.00 0.00
toilet_flush 16 0.75 0.75 0.75
train 9 0.32 0.78 0.45
vacuum_cleaner 8 0.78 0.88 0.82
washing_machine 9 0.26 0.78 0.39
water_drops 7 0.17 0.29 0.21
wind 5 0.14 0.60 0.22
"""

SISDR_V28 = """
air_conditioner 17 44.28 -8.71 -52.99
airplane 11 14.01 5.76 -8.25
breathing 14 0.19 -15.22 -15.41
brushing_teeth 13 17.01 -9.61 -26.62
can_opening 15 13.37 -3.76 -17.13
car_horn 21 38.56 3.70 -34.86
cat 9 51.24 -4.04 -55.28
chainsaw 7 15.34 1.51 -13.84
children_playing 17 8.11 -20.13 -28.24
chirping_birds 9 3.38 -11.54 -14.92
church_bells 14 26.71 11.27 -15.44
clapping 14 21.26 -3.08 -24.34
clock_alarm 9 14.50 13.26 -1.25
clock_tick 13 13.91 0.01 -13.89
coughing 11 5.12 -15.92 -21.04
cow 16 14.70 0.17 -14.53
crackling_fire 12 2.35 -11.99 -14.34
crickets 14 22.48 -1.78 -24.26
crow 10 14.01 6.12 -7.89
crying_baby 17 35.58 2.21 -33.38
dog 14 5.03 -13.03 -18.06
door_wood_creaks 18 12.45 -22.10 -34.55
door_wood_knock 7 43.06 17.96 -25.11
drilling 16 34.18 -5.92 -40.10
drinking_sipping 9 25.97 -10.98 -36.96
engine 12 41.24 7.44 -33.80
fireworks 14 37.08 -2.97 -40.05
footsteps 21 33.82 2.18 -31.64
frog 20 19.87 6.31 -13.56
glass_breaking 6 42.08 9.23 -32.86
gun_shot 14 0.94 -11.93 -12.86
hand_saw 7 10.15 -0.57 -10.72
helicopter 11 18.81 0.19 -18.62
hen 17 44.03 9.62 -34.41
insects 14 20.73 -11.94 -32.67
jackhammer 25 31.06 0.25 -30.80
keyboard_typing 14 28.63 1.20 -27.43
laughing 13 25.18 -6.60 -31.78
mouse_click 10 8.53 -24.78 -33.31
pig 13 9.07 -5.84 -14.91
pouring_water 16 36.51 9.25 -27.26
rain 17 13.52 -0.44 -13.96
rooster 7 25.48 -1.56 -27.03
sea_waves 14 29.08 2.64 -26.44
sheep 8 50.50 8.52 -41.98
siren 9 20.29 2.87 -17.42
sneezing 9 -6.42 -18.64 -12.21
snoring 17 35.19 -0.18 -35.37
street_music 18 15.93 -6.36 -22.29
thunderstorm 9 32.13 19.00 -13.14
toilet_flush 19 17.57 -1.70 -19.27
train 15 27.15 8.83 -18.32
vacuum_cleaner 15 33.01 8.48 -24.52
washing_machine 13 44.18 10.43 -33.76
water_drops 11 -6.18 -20.21 -14.03
wind 12 14.59 0.90 -13.69
"""

TRAINING_V28 = """
1 0.4758 0.6756 0.8136
2 0.4752 0.6337 0.7920
3 0.4685 0.6375 0.7872
4 0.4553 0.6421 0.7764
5 0.4997 0.6295 0.8145
6 0.4654 0.6705 0.8006
7 0.4433 0.6181 0.7523
8 0.4465 0.6223 0.7577
9 0.4323 0.6143 0.7394
10 0.4286 0.6083 0.7327
11 0.4388 0.6067 0.7421
12 0.4158 0.5954 0.7135
13 0.4288 0.5995 0.7286
14 0.4254 0.5991 0.7250
15 0.4260 0.5968 0.7244
16 0.4156 0.5810 0.7061
17 0.4055 0.5891 0.7000
18 0.3975 0.5682 0.6816
19 0.4051 0.5681 0.6891
20 0.3982 0.5638 0.6801
21 0.4105 0.5528 0.6869
22 0.3889 0.5589 0.6683
23 0.3865 0.5417 0.6573
24 0.3910 0.5776 0.6798
25 0.4178 0.5531 0.6943
26 0.3861 0.5443 0.6583
27 0.3851 0.5427 0.6564
28 0.3848 0.5431 0.6564
29 0.4013 0.5530 0.6778
30 0.3828 0.5406 0.6531
31 0.3726 0.5481 0.6467
32 0.3640 0.5323 0.6301
33 0.3605 0.5274 0.6242
34 0.3574 0.5252 0.6200
35 0.3669 0.5174 0.6255
36 0.3552 0.5169 0.6136
37 0.3462 0.5119 0.6021
38 0.3729 0.5404 0.6431
39 0.3482 0.4944 0.5954
40 0.3397 0.5139 0.5967
41 0.3573 0.4982 0.6064
42 0.3347 0.4878 0.5786
43 0.3392 0.4695 0.5740
44 0.3219 0.4904 0.5671
45 0.3482 0.5008 0.5986
46 0.3272 0.4843 0.5694
47 0.3398 0.4964 0.5880
48 0.3452 0.4959 0.5931
49 0.3225 0.4715 0.5582
50 0.3115 0.4528 0.5379
51 0.3127 0.4683 0.5469
52 0.3060 0.4560 0.5340
53 0.2996 0.4583 0.5288
54 0.2987 0.4496 0.5235
55 0.3045 0.4627 0.5358
56 0.2928 0.4495 0.5176
57 0.2942 0.4507 0.5195
58 0.2892 0.4504 0.5144
59 0.3000 0.4430 0.5215
60 0.2821 0.4439 0.5041
"""


def _read(block, names):
    return pd.read_csv(io.StringIO(block.strip()), sep=r"\s+",
                       header=None, names=names)


det_df = _read(DETECTION_V28, ["class", "N", "precision", "recall", "f1"])
sisdr_df = _read(SISDR_V28, ["class", "N", "si_sdr_mix_db",
                             "si_sdr_model_db", "si_sdri_db"])
train_df = _read(TRAINING_V28, ["epoch", "val_separation_l1",
                                "val_detection_bce", "val_total"])

print(f"Loaded: {len(summary_df)} versions, "
      f"{len(det_df)} detection rows, {len(sisdr_df)} SI-SDR rows, "
      f"{len(train_df)} training epochs.")
summary_df

## 3. Figures

### Figure 1 - Detection F1 across versions
The central result: F1 rises from 0.02 (v2.3, when FSD50K flooded the
candidate pool with phantom classes) to **0.32** at v2.8, after removing
FSD50K and reverting the collapsed focal loss to BCE.

In [ ]:
# Figure 1 — Detection F1 across model versions (the headline result).
f1 = summary_df.dropna(subset=["detection_f1"]).copy()
fig, ax = plt.subplots(figsize=(9, 5))
colors = [HILITE if v == "v2.8" else ACCENT for v in f1["version"]]
bars = ax.bar(f1["version"], f1["detection_f1"], color=colors)
for b, val in zip(bars, f1["detection_f1"]):
    ax.text(b.get_x() + b.get_width() / 2, val + 0.005,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Macro F1")
ax.set_xlabel("Model version")
ax.set_title("Detection F1 across model versions\n(higher is better)")
ax.set_ylim(0, max(f1["detection_f1"]) * 1.2)
ax.annotate("FSD50K added\n(phantom FPs)", xy=(2, 0.02), xytext=(2, 0.13),
            ha="center", fontsize=8, color=MUTED,
            arrowprops=dict(arrowstyle="->", color=MUTED))
ax.annotate("focal-loss\ncollapse", xy=(5, 0.0), xytext=(5, 0.10),
            ha="center", fontsize=8, color=MUTED,
            arrowprops=dict(arrowstyle="->", color=MUTED))
save(fig, "fig1_detection_f1_by_version")
plt.show()

### Figure 2 - SI-SDRi across versions
All values are negative: many test mixtures are already dominated by
the target class, so spectrogram masking cannot improve the
waveform-level SI-SDR. This is a known limitation of mask-based
separation with mixture-phase reuse, discussed in the thesis.

In [ ]:
# Figure 2 — Separation SI-SDRi across model versions.
si = summary_df.dropna(subset=["si_sdri_db"]).copy()
fig, ax = plt.subplots(figsize=(9, 5))
colors = [HILITE if v == "v2.8" else ACCENT for v in si["version"]]
bars = ax.bar(si["version"], si["si_sdri_db"], color=colors)
for b, val in zip(bars, si["si_sdri_db"]):
    ax.text(b.get_x() + b.get_width() / 2, val - 0.6,
            f"{val:.1f}", ha="center", va="top", fontsize=9)
ax.set_ylabel("SI-SDRi (dB)")
ax.set_xlabel("Model version")
ax.set_title("Separation SI-SDRi across model versions\n"
             "(closer to 0 is better; all negative — see thesis discussion)")
ax.axhline(0, color="black", linewidth=0.8)
save(fig, "fig2_sisdri_by_version")
plt.show()

### Figure 3 - v2.8 training curves
The detection BCE (red) falls from 0.68 to 0.44 over 60 epochs - proof
the head learned. In v2.7 the focal loss kept this term flat near 0.07
from initialisation (its gradient is ~10x smaller than BCE at p=0.5),
so the head never trained.

In [ ]:
# Figure 3 — v2.8 training curves: the detection head actually learns.
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_df["epoch"], train_df["val_total"], label="Total val loss",
        color="black", linewidth=2)
ax.plot(train_df["epoch"], train_df["val_separation_l1"],
        label="Separation L1 (val)", color=ACCENT, linewidth=1.5)
ax.plot(train_df["epoch"], train_df["val_detection_bce"],
        label="Detection BCE (val)", color=HILITE, linewidth=1.5)
ax.axvline(48, color=MUTED, linestyle="--", linewidth=1)
ax.text(48.3, ax.get_ylim()[1] * 0.96, "LR /2", color=MUTED, fontsize=8)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation loss")
ax.set_title("v2.8 training - detection BCE steadily decreases\n"
             "(v2.7 focal loss stayed flat at ~0.07: gradient collapse)")
ax.legend()
save(fig, "fig3_v28_training_curves")
plt.show()

### Figure 4 - v2.8 detection F1 per class
Strong classes (`vacuum_cleaner`, `toilet_flush`, `crying_baby`,
`gun_shot`) sit well above the 0.32 mean; broadband or very short
classes (`air_conditioner`, `thunderstorm`, `chainsaw`) sit at zero.

In [ ]:
# Figure 4 — v2.8 per-class detection F1 (sorted, colour-graded).
d = det_df.sort_values("f1", ascending=True)
fig, ax = plt.subplots(figsize=(8, 13))
cmap = plt.cm.RdYlGn
bar_colors = cmap(d["f1"] / max(d["f1"].max(), 1e-9))
ax.barh(d["class"], d["f1"], color=bar_colors)
ax.axvline(det_df["f1"].mean(), color="black", linestyle="--", linewidth=1,
           label=f"mean F1 = {det_df['f1'].mean():.2f}")
ax.set_xlabel("F1")
ax.set_title("v2.8 detection F1 by class (200 mixtures)")
ax.legend(loc="lower right")
ax.margins(y=0.01)
save(fig, "fig4_v28_detection_f1_per_class")
plt.show()

### Figure 5 - v2.8 SI-SDRi per class
`clock_alarm` is nearly break-even (-1.25 dB); classes whose mixtures
start at very high SI-SDR (`cat`, `air_conditioner`) show the most
negative improvement because there was nothing to separate.

In [ ]:
# Figure 5 — v2.8 per-class SI-SDRi (sorted).
s = sisdr_df.sort_values("si_sdri_db", ascending=True)
fig, ax = plt.subplots(figsize=(8, 13))
bar_colors = ["#16a34a" if v > -15 else ACCENT if v > -30 else HILITE
              for v in s["si_sdri_db"]]
ax.barh(s["class"], s["si_sdri_db"], color=bar_colors)
ax.axvline(sisdr_df["si_sdri_db"].mean(), color="black", linestyle="--",
           linewidth=1, label=f"mean = {sisdr_df['si_sdri_db'].mean():.1f} dB")
ax.set_xlabel("SI-SDRi (dB)")
ax.set_title("v2.8 SI-SDRi by class (800 mixtures)")
ax.legend(loc="lower left")
ax.margins(y=0.01)
save(fig, "fig5_v28_sisdri_per_class")
plt.show()

### Figure 6 - v2.8 precision vs recall
Each point is a class (size = number of positive mixtures). Points
toward the top-right are the well-detected classes; the dotted lines
are F1 iso-curves.

In [ ]:
# Figure 6 — v2.8 detection precision vs recall, per class.
fig, ax = plt.subplots(figsize=(8, 7))
sizes = det_df["N"] * 12
sc = ax.scatter(det_df["recall"], det_df["precision"], s=sizes,
                c=det_df["f1"], cmap="viridis", alpha=0.75,
                edgecolors="white", linewidths=0.5)
for _, r in det_df.iterrows():
    if r["f1"] >= 0.55 or (r["precision"] < 0.15 and r["recall"] > 0.4):
        ax.annotate(r["class"], (r["recall"], r["precision"]),
                    fontsize=7, alpha=0.8,
                    xytext=(3, 3), textcoords="offset points")
rr = np.linspace(0.01, 1, 200)
for f1v in (0.2, 0.4, 0.6, 0.8):
    pp = (f1v * rr) / (2 * rr - f1v)
    mask = (pp > 0) & (pp <= 1)
    ax.plot(rr[mask], pp[mask], color=MUTED, linewidth=0.6, linestyle=":")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.set_title("v2.8 detection: precision vs recall per class\n"
             "(point size = #positives; dotted = F1 iso-curves)")
fig.colorbar(sc, ax=ax, label="F1")
save(fig, "fig6_v28_precision_recall")
plt.show()

## 4. Excel workbook
One `.xlsx` with four sheets, written next to the figures on Drive.

In [ ]:
# Write every table to one multi-sheet Excel workbook on Drive.
xlsx = TABLES / "snc_results.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as xl:
    summary_df.to_excel(xl, sheet_name="Version comparison", index=False)
    det_df.to_excel(xl, sheet_name="v2.8 detection", index=False)
    sisdr_df.to_excel(xl, sheet_name="v2.8 SI-SDR", index=False)
    train_df.to_excel(xl, sheet_name="v2.8 training curve", index=False)
print("wrote", xlsx)

## 5. Done - verify saved artifacts

In [ ]:
# Confirm everything landed on Drive.
print("Artifacts under", OUT, "\n")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}   ({p.stat().st_size // 1024} KB)")